In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

In [ ]:
data = pd.read_csv("C:\\Users\\mayan\\OneDrive\\Desktop\\,\\PERSONAL\\Project 1- Semiconductor Error detection\\Unsupervised Learning Basics Scikit\\Advertising_dataset.csv", index_col = 0)
print(data)

In [ ]:
feature_cols = ['TV','radio','newspaper']
x = data[feature_cols]
y = data.sales

In [ ]:
# 10 fold  cross validation with all 3 features
lm = LinearRegression()
mse_score = cross_val_score(lm, x, y, cv=10, scoring="neg_mean_squared_error")
# cross_val_score -> CAN BE USED FOR BOTH, CLASSIFICATION AND REGRESSION MODELS BOTH
# scoring="neg_mean_squared_error" -> THIS TIME WE CANT USE ACCURACY AS OUR EVALUATION METRIC AS IT IS ONLY RELEVANT FOR CLASSIFICCATION PROBLEMS SO WE ARE TRYING TO USE ROOT MEAN SQUARE ERROR
print(mse_score)

In [ ]:
# converting the mse_scores from negative to positive 
mse_score = -mse_score
print(mse_score)

In [ ]:
rmse_score = np.sqrt(mse_score)
print(rmse_score)
print (rmse_score.mean())

### NOW WE CALCULATE WHEN WE DO NOT ADD NEWSPAPER AS A FEATURE 

In [ ]:
feature_cols = ['TV','radio']
x = data[feature_cols]
y = data.sales

lm = LinearRegression()
mse_score = -cross_val_score(lm, x, y, cv=10, scoring="neg_mean_squared_error")
rmse_score = np.sqrt(mse_score)
print(rmse_score)
print (rmse_score.mean())

## Improvements to Cross-Validation
#### Repeated Cross-Validation
- Perform cross-validation several times using distinct random data partitions each time, then compute the overall average.
- Why it helps: It lowers the variance tied to a single cross-validation run, producing a far more dependable estimate of how the model performs on unseen data.

#### Creating a Hold-Out Set
- Reserve a segment of your dataset prior to starting any model construction or tuning.
- Identify your top-performing model by running cross-validation exclusively on the remaining training data, and then evaluate that chosen model on your reserved test set.
- Why it helps: The hold-out set remains completely untouched throughout model selection, serving as a genuine test of out-of-sample generalization.

#### Feature Engineering and Selection Within Cross-Validation Iterations
- Standard practice often applies feature selection or transformation across the entire dataset prior to cross-validation.
- Instead, embed all feature processing and selection steps inside each individual cross-validation loop.
- Why it helps: Prevents data leakage and offers a more realistic assessment of performance by closely imitating how the pipeline encounters brand-new data.

## LASSO (L1) / RIDGE (L2) REGRESSION

In [ ]:
data = pd.read_csv("C:\\Users\\mayan\\OneDrive\\Desktop\\,\\PERSONAL\\Project 1- Semiconductor Error detection\\Unsupervised Learning Basics Scikit\\Melbourne_housing_dataset.csv")

# using limited columns which makes more sense for serving our purpose
cols_to_use = ['Suburb', 'Rooms', 'Type', 'Method', 'SellerG', 'Regionname', 'Propertycount', 
               'Distance', 'CouncilArea', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'Price']
data = data[cols_to_use]
data.isna().sum()  #Tells us how much NaN data we  have

In [ ]:
# DATA CLEANING (NEEDS HUMAN JUDGEMENT)
# Regionname 3, Propertycount 3, Distance 1, CouncilArea 3 -> LESS VALUES, CAN BE DROPPED
# Bedroom2, Bathroom, Car -> IF THERE IS NO DATA THEN THAT MEANS THAT THE VALUE IS 0
# Landsize, BuildingArea -> WE CAN TEAKE AVERAGE AND REPLACE NAN VALUES
# Price -> IS THE DATA WE ARE ESTIMATING SO WE HAVE TO DROP NAN VALUES

cols_to_fill_zero = ['Bedroom2', 'Bathroom', 'Car']
data[cols_to_fill_zero] = data[cols_to_fill_zero].fillna(0)

data['Landsize'] = data['Landsize'].fillna(data['Landsize'].mean())
data['BuildingArea'] = data['BuildingArea'].fillna(data['BuildingArea'].mean())

data = data.dropna() # remaining data is to be dropped

x = data[['Suburb', 'Rooms', 'Type', 'Method', 'SellerG', 'Regionname', 'Propertycount', 'Distance', 'CouncilArea', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea']]
x = pd.get_dummies(x, drop_first=True)
# makes all the stringvalues as usable data
y = data['Price']
data.isna().sum()

In [ ]:
# USING LINEAR REGRESSION MODEL
from sklearn.model_selection import train_test_split


train_x, test_x, train_y, test_y = train_test_split(x, y, test_size=0.3, random_state=2)
from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(train_x, train_y)
print(reg.score(test_x, test_y))
print(reg.score(train_x, train_y))

In [ ]:
# USING LASSO (L1 REGULARISED) REGRESSION MODEL
from sklearn import linear_model
lasso_reg = linear_model.Lasso(alpha=50, max_iter=10000, tol=0.1)
lasso_reg.fit(train_x, train_y)
print(lasso_reg.score(test_x, test_y))
print(lasso_reg.score(train_x, train_y))

In [ ]:
# USING RIDGE (L2 REGULARISED) REGRESSION MODEL
from sklearn.linear_model import Ridge
ridge_reg = Ridge(alpha=50, max_iter=100, tol=0.1)
ridge_reg.fit(train_x, train_y)
print(ridge_reg.score(test_x, test_y))
print(ridge_reg.score(train_x, train_y))

In [ ]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.dummy import DummyRegressor

data = data.dropna(subset=['Price'])        # only drop rows with no price
y = np.log(data['Price'])                   # log makes prices less skewed
X = data.drop(columns='Price')

num_cols = X.select_dtypes(exclude='object').columns
cat_cols = X.select_dtypes(include='object').columns

prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

models = {
    "Mean baseline": DummyRegressor(),
    "Linear": LinearRegression(),
    "Ridge": RidgeCV(alphas=np.logspace(-2, 4, 13)),
    "Lasso": LassoCV(cv=5, max_iter=20000),
}

cv = KFold(n_splits=5, shuffle=True, random_state=2)
for name, m in models.items():
    pipe = make_pipeline(prep, m)
    mae = -cross_val_score(pipe, X, y, cv=cv, scoring="neg_mean_absolute_error")
    print(f"{name:14s} MAE = {mae.mean():.3f} ± {mae.std():.3f}")